# Notebook 05 — Hypothesis Testing Fundamentals

**Module:** 03 — Statistics and Probability  
**Notebook:** 05 of 18  
**Prerequisites:** NB04  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Hypothesis testing is the formal framework for deciding whether an observed effect
could be explained by random chance. It underlies every differential expression
analysis, every GWAS, every clinical trial. The framework is elegant but frequently
misused — this notebook builds it from scratch so you can recognize errors in
published papers and avoid them in your own work.

---
## Step 2 — Intuition

The null hypothesis ($H_0$) is the boring, conservative claim: 'there is no effect.'
The alternative hypothesis ($H_1$) is your scientific claim: 'there is an effect.'

A hypothesis test asks: 'if $H_0$ were true, how surprising would my observed data be?'
If the data would be very surprising under $H_0$ (small p-value), you reject $H_0$.
If not surprising, you fail to reject — you have no evidence either way.

**Critical: failing to reject $H_0$ does not mean $H_0$ is true.**

---
## Step 3 — Biological Background

**Differential expression framing:**
- $H_0$: gene X has the same mean expression in tumour and normal tissue
- $H_1$: gene X has different mean expression
- Data: log2CPM values from $n$ tumour and $m$ normal samples
- Test statistic: t-statistic (NB06)
- Decision: reject $H_0$ if $|t|$ is large enough (p < α)

**GWAS framing:**
- $H_0$: SNP rs12345 has no association with disease risk
- Test: logistic regression; p-value from likelihood ratio test
- Threshold: typically p < 5 × 10⁻⁸ (Bonferroni for ~1M SNPs)

**Type I and Type II errors:**

|  | $H_0$ true | $H_0$ false |
|--|-----------|-------------|
| Reject $H_0$ | **Type I error** (false positive, rate = α) | Correct (True Positive) |
| Fail to reject | Correct (True Negative) | **Type II error** (false negative, rate = β) |

---
## Step 4 — Mathematical Explanation

**p-value (correct definition):**
> The probability of observing a test statistic at least as extreme as the one computed,
> **assuming $H_0$ is true.**

$$p = P(|T| \geq |t_{\text{obs}}| \mid H_0)$$

**What a p-value is NOT:**
- It is NOT the probability that $H_0$ is true
- It is NOT the probability of obtaining the data by chance
- It does NOT measure the effect size
- A small p-value does NOT mean the effect is large

**Statistical power:** $1 - \beta = P(\text{reject } H_0 \mid H_1 \text{ true})$

**One-tailed vs. two-tailed:**
- Two-tailed: $H_1$: $\mu \neq \mu_0$ — test for any difference
- One-tailed: $H_1$: $\mu > \mu_0$ — test for specific direction
- **Default to two-tailed** unless there is a strong prior reason for directionality

---
## Step 5 — Computational Explanation

**Permutation test — non-parametric hypothesis test from scratch:**
1. Compute the observed test statistic $t_{\text{obs}}$ (e.g. difference in means)
2. Randomly shuffle the group labels 10,000 times
3. Recompute the test statistic under each shuffle → null distribution
4. p-value = fraction of null statistics ≥ $|t_{\text{obs}}|$

This requires no distributional assumption and is always valid. It is the gold standard
for understanding what a p-value means mechanically.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Permutation test from scratch: the mechanistic p-value
rng = np.random.default_rng(42)

# Simulate two groups: control and treatment gene expression
control = rng.normal(loc=5.0, scale=1.2, size=15)
treatment = rng.normal(loc=6.0, scale=1.2, size=15)  # true difference = 1.0

def permutation_test(a: np.ndarray, b: np.ndarray,
                      n_permutations: int = 10_000,
                      rng=None) -> dict:
    """
    Permutation test for difference in means.

    Returns
    -------
    dict with: observed_diff, null_distribution, p_value
    """
    if rng is None:
        rng = np.random.default_rng()
    observed_diff = np.abs(a.mean() - b.mean())
    combined = np.concatenate([a, b])
    na = len(a)
    null_diffs = np.empty(n_permutations)
    for i in range(n_permutations):
        shuffled = rng.permutation(combined)
        null_diffs[i] = np.abs(shuffled[:na].mean() - shuffled[na:].mean())
    p_value = (null_diffs >= observed_diff).mean()
    return dict(observed_diff=observed_diff, null_distribution=null_diffs, p_value=p_value)

result = permutation_test(control, treatment, rng=rng)
print(f"Observed |difference in means|: {result['observed_diff']:.4f}")
print(f"Permutation p-value: {result['p_value']:.4f}")

# Compare to parametric t-test
t_stat, p_ttest = stats.ttest_ind(control, treatment)
print(f"Parametric t-test p-value: {p_ttest:.4f}")

In [ ]:
# Cell 6.2 — Type I error rate: does α really control false positives?
rng2 = np.random.default_rng(0)
alpha = 0.05
n_trials = 10_000
n_per_group = 20

false_positives = 0
for _ in range(n_trials):
    # Both groups drawn from the SAME distribution (H0 is TRUE)
    a = rng2.normal(0, 1, n_per_group)
    b = rng2.normal(0, 1, n_per_group)
    _, p = stats.ttest_ind(a, b)
    if p < alpha:
        false_positives += 1

observed_type1 = false_positives / n_trials
print(f"Type I error rate across {n_trials:,} trials (H0 always true):")
print(f"  Observed: {observed_type1:.4f}")
print(f"  Nominal α: {alpha:.4f}")
print(f"  Match: {'YES' if abs(observed_type1 - alpha) < 0.01 else 'NO'}")

In [ ]:
# Cell 6.3 — Power: how often do we correctly detect a real effect?
effect_sizes = [0.2, 0.5, 0.8, 1.2]  # Cohen's d
sample_sizes = [5, 10, 20, 50, 100]
n_sim = 2000

print(f"Statistical power (proportion of true effects detected at α={alpha}):")
print(f"{'n':>6} {'d=0.2':>8} {'d=0.5':>8} {'d=0.8':>8} {'d=1.2':>8}")
print("-" * 46)
for n in sample_sizes:
    powers = []
    for d in effect_sizes:
        detected = sum(
            stats.ttest_ind(
                rng.normal(0, 1, n), rng.normal(d, 1, n)
            )[1] < alpha
            for _ in range(n_sim)
        )
        powers.append(detected / n_sim)
    print(f"  {n:>4}  " + "  ".join(f"{p:>6.2f}" for p in powers))

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Permutation null distribution and power-sample size curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: null distribution from permutation test
ax = axes[0]
null = result['null_distribution']
obs = result['observed_diff']
ax.hist(null, bins=60, color="steelblue", edgecolor="none", density=True, label="Null distribution")
ax.axvline(obs, color="tomato", lw=2, label=f"Observed |Δmean| = {obs:.3f}")
ax.fill_between(np.linspace(obs, null.max(), 100),
                np.zeros(100), 0.05, color="tomato", alpha=0.2,
                label=f"p = {result['p_value']:.4f}")
ax.set_xlabel("|Difference in means|"); ax.set_ylabel("Density")
ax.set_title("Permutation test: null distribution")
ax.legend(fontsize=8)

# Panel 2: power curves
ax = axes[1]
colors_d = ["steelblue", "green", "orange", "tomato"]
n_vals_plot = [5, 10, 20, 50, 100]
power_curves = {d: [] for d in effect_sizes}

rng3 = np.random.default_rng(7)
for d in effect_sizes:
    for n in n_vals_plot:
        detected = sum(
            stats.ttest_ind(rng3.normal(0,1,n), rng3.normal(d,1,n))[1] < alpha
            for _ in range(1000)
        )
        power_curves[d].append(detected / 1000)

for d, color in zip(effect_sizes, colors_d):
    ax.plot(n_vals_plot, power_curves[d], 'o-', color=color, lw=1.5, label=f"d={d}")
ax.axhline(0.8, color="black", ls="--", lw=1, label="80% power target")
ax.set_xlabel("n per group"); ax.set_ylabel("Power (1-β)")
ax.set_title("Power vs. sample size")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `permutation_test()` from scratch without looking at Cell 6.1.
   Apply it to test whether two GC content distributions are different.
2. The Cell 6.2 simulation confirms that the t-test controls Type I error at α=0.05.
   Repeat the simulation with n_per_group=3. Does the t-test still control Type I error?
   What about n=3 when the data is Exponential (non-normal)?
3. From the power table in Cell 6.3: what n is needed to detect an effect size d=0.5
   with 80% power at α=0.05? Estimate from the table, then compute analytically using
   `scipy.stats.norm`'s percent-point function.
4. Give three examples of a Type II error (missed detection) in genomics. Why does
   underpowering a study cause this?

---
## Quiz — Active Recall

1. Write the correct definition of a p-value from memory.
2. What is a Type I error? Type II error? Which does α control?
3. What does 'fail to reject H₀' mean? What does it NOT mean?
4. What is the difference between a one-tailed and two-tailed test? When should you use each?
5. If power is 80%, what does that mean in plain English for a gene expression study?

---
## Reflection

**Date completed:** ____________________

> *[Could you explain a p-value correctly to a colleague without using the phrase 'probability the result is by chance'?]*

---
*Next: `06_t_tests.ipynb`*